# 4.3 · Subagentes

**Problema:** el agente principal lee incidencias, artículos y resultados largos. Su contexto se llena de ruido y empieza a decidir peor. Además, clasificar la prioridad es una tarea concreta que podría hacer un especialista más barato.

**Al terminar tendrás:** un subagente `clasificador` que recibe una incidencia y devuelve prioridad y categoría.

## Preparación

Herramientas de `taller_utils.py` (4.1) y el servidor de la base de conocimiento (4.2).

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import herramientas
from taller_utils import imprimir, buscar_incidencias, crear_incidencia, servidor_kb
from claude_agent_sdk import (
    query, create_sdk_mcp_server, ClaudeAgentOptions, AgentDefinition,
    AssistantMessage, ResultMessage,
)

soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia])
HERRAMIENTAS_KB = ["mcp__kb__list_directory", "mcp__kb__read_text_file", "mcp__kb__search_files"]

In [ ]:
CORREO = "Llega este correo de Initech: 'nadie de mi equipo puede entrar desde esta mañana'. Regístralo."

In [ ]:
SYSTEM = ("Eres un agente de soporte de nivel 1. Respondes en español y de forma breve. "
          "Antes de crear una incidencia, comprueba que no existe ya una abierta sobre el mismo problema. "
          "La prioridad de cada incidencia se asigna según los criterios oficiales de prioridad: no la supongas.")

## Sin subagente

Primero, el agente haciéndolo todo solo, con la base de conocimiento conectada. Para decidir la prioridad tiene que buscar y leer los criterios (`prioridades.md`), y todo eso se queda en **su** contexto.

In [ ]:
herramientas.restaurar_datos()

options_solo = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte, "kb": servidor_kb()},
    tools=[],
    allowed_tools=["mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia", *HERRAMIENTAS_KB],
    max_turns=15,
    setting_sources=[],
)

async for msg in query(prompt=CORREO, options=options_solo):
    imprimir(msg)
    if isinstance(msg, ResultMessage):
        resultado_solo = msg

El agente está haciendo de todo. Vamos a darle un compañero especializado.

## Definir el subagente

Un subagente se define con `AgentDefinition` y se registra en `agents`. Tiene cuatro piezas:

| Pieza | Qué es |
|---|---|
| `description` | **cuándo usarlo**. La lee el agente principal para decidir si delega. |
| `prompt` | su system prompt |
| `tools` | **solo** las herramientas que necesita |
| `model` | puede ser un modelo más pequeño y barato |

El agente principal delega con una herramienta integrada, que hay que darle (`tools`) y permitir (`allowed_tools`): `Task`.

Dos detalles:

- `env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"}`: el SDK puede lanzar los subagentes **en segundo plano**, y entonces el principal sigue trabajando sin esperar su respuesta. Para este flujo (clasificar → crear) queremos que espere, así que desactivamos las tareas en segundo plano.
- El servidor de la base de conocimiento ya **no** está en el agente principal. Se lo damos solo al subagente, con `mcpServers`. El principal ni siquiera ve la KB: para aplicar los criterios de prioridad, tiene que delegar. Y su contexto no se llena con artículos que no necesita.

> Ojo: los campos de `AgentDefinition` que tienen varias palabras van en **camelCase** (`disallowedTools`, `maxTurns`…), no en snake_case como `ClaudeAgentOptions`.

In [ ]:
options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},   # el principal ya no tiene la KB
    tools=["Task"],   # la herramienta integrada con la que se delega en subagentes
    allowed_tools=["Task", "mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia",
                   *HERRAMIENTAS_KB],
    agents={
        "clasificador": AgentDefinition(
            description=("Especialista en clasificar incidencias de soporte. Úsalo cuando haya que "
                         "decidir la prioridad o la categoría de una incidencia nueva o existente."),
            prompt=("Eres un clasificador de incidencias. Dada una incidencia, consulta los criterios de "
                    "prioridad (prioridades.md en la base de conocimiento) y devuelve SOLO: prioridad "
                    "(alta/media/baja), categoría (facturación/técnico/cuenta/otro) y una justificación "
                    "de una línea."),
            tools=HERRAMIENTAS_KB,
            mcpServers=[{"kb": servidor_kb()}],   # la KB es solo del subagente
            model="claude-haiku-5-5",
        ),
    },
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},   # que el principal espere al subagente
    max_turns=15,
    setting_sources=[],
)

## Verlo delegar

La misma tarea. Guardamos los mensajes en una lista para analizarlos después sin volver a ejecutar el agente.

In [ ]:
herramientas.restaurar_datos()

mensajes = []
async for msg in query(prompt=CORREO, options=options):
    mensajes.append(msg)
    imprimir(msg)
resultado_con = mensajes[-1]

En las trazas: el agente principal invoca la herramienta de delegación (en las trazas aparece como `Agent`), el subagente trabaja (líneas `[subagente]`) y devuelve un resumen corto, y el principal crea la incidencia con esa prioridad.

### ¿Quién hace qué?

Los mensajes del subagente llevan `parent_tool_use_id`: el id de la llamada de delegación de la que cuelgan. Los del agente principal lo tienen a `None`.

In [ ]:
for m in mensajes:
    if isinstance(m, AssistantMessage):
        quien = "subagente" if m.parent_tool_use_id else "principal"
        print(f"{quien:10} {m.model:20} {[type(b).__name__ for b in m.content]}")

## Por qué funciona

- **Contexto aislado:** el subagente lee los criterios enteros, pero al principal solo le llega la conclusión. Compara las trazas: sin subagente, las lecturas de la KB eran del principal.
- **Mínimo privilegio:** el clasificador no puede crear ni cerrar incidencias, y el principal no puede leer la KB.
- **Coste:** el clasificador corre en un modelo más pequeño. `model_usage` desglosa el coste por modelo:

In [ ]:
print(f"sin subagente: ${resultado_solo.total_cost_usd:.4f}  ({resultado_solo.num_turns} turnos)")
print(f"con subagente: ${resultado_con.total_cost_usd:.4f}  ({resultado_con.num_turns} turnos del principal)")
{modelo: round(uso["costUSD"], 4) for modelo, uso in resultado_con.model_usage.items()}

En una tarea tan corta, la diferencia de coste puede ir en cualquier sentido: delegar también cuesta (el subagente arranca con su propio contexto). La ganancia se nota cuando la subtarea lee mucho: todo eso se queda fuera del contexto del principal, que se mantiene limpio durante toda la conversación.

**Cuándo no usar subagentes:** tareas cortas en las que el coste de delegar supera al de hacerlo directamente.

In [ ]:
herramientas.restaurar_datos()

## Resumen

- Un subagente es un especialista: `description` (cuándo), `prompt`, `tools` mínimas y `model` adecuado.
- El principal delega con la herramienta `Task`; los mensajes del subagente llevan `parent_tool_use_id`.
- Aísla el contexto, aplica mínimo privilegio y puede abaratar. En tareas cortas, no compensa.

`CLASIFICADOR` queda en `taller_utils.py`.

**Para el siguiente vídeo:** ahora el agente decide delegar. ¿Cómo sé cuándo y por qué lo ha hecho? Necesitamos observarlo.

## 🔸 Reto

Cambia la `description` del subagente a algo vago (`"Ayuda con incidencias."`) y observa si deja de usarlo o lo usa de más. Prueba con el correo de Initech y con una pregunta que no necesita clasificar nada, como "¿Qué incidencias abiertas tiene ACME?".

### Solución

Solo cambia la `description`:

In [ ]:
options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},   # el principal ya no tiene la KB
    tools=["Task"],   # la herramienta integrada con la que se delega en subagentes
    allowed_tools=["Task", "mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia",
                   *HERRAMIENTAS_KB],
    agents={
        "clasificador": AgentDefinition(
            description="Ayuda con incidencias.",
            prompt=("Eres un clasificador de incidencias. Dada una incidencia, consulta los criterios de "
                    "prioridad (prioridades.md en la base de conocimiento) y devuelve SOLO: prioridad "
                    "(alta/media/baja), categoría (facturación/técnico/cuenta/otro) y una justificación "
                    "de una línea."),
            tools=HERRAMIENTAS_KB,
            mcpServers=[{"kb": servidor_kb()}],   # la KB es solo del subagente
            model="claude-haiku-5-5",
        ),
    },
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},   # que el principal espere al subagente
    max_turns=15,
    setting_sources=[],
)

In [ ]:
async def delegaciones(prompt):
    herramientas.restaurar_datos()
    n = 0
    async for msg in query(prompt=prompt, options=options):
        imprimir(msg)
        if isinstance(msg, AssistantMessage) and not msg.parent_tool_use_id:
            n += sum(1 for b in msg.content if getattr(b, "name", "") in ("Agent", "Task"))
    print(f"\n→ delegaciones: {n}")

In [ ]:
await delegaciones(CORREO)

In [ ]:
await delegaciones("¿Qué incidencias abiertas tiene ACME?")

Con una descripción vaga, el agente principal no sabe **cuándo** delegar ni qué esperar del subagente. Aquí el subagente es la única vía para llegar a los criterios de prioridad, así que puede que lo siga usando para clasificar; pero ya no es porque la descripción se lo diga, y en otras tareas puede dejar de usarlo o usarlo de más (para listar incidencias, por ejemplo, que el subagente ni siquiera puede hacer). Ejecuta las celdas varias veces: el resultado cambia de una ejecución a otra. La descripción es lo único que el principal sabe del subagente.

Es la misma lección que con las herramientas: **la descripción es un prompt**.

In [ ]:
herramientas.restaurar_datos()